# 06 — Final Evaluation

**Goal (game plan Week 8):** evaluate the final model with MAPE / MdAPE, break errors down by price band and region, and save `docs/metrics_summary.csv`.
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

What changed in Week 8, before this evaluation:
- **Wider tuning** of every model (`python src/models.py --tuned`). The Week 5 and Week 7 choices sat on the edge of their grids, so the search was widened around them. LightGBM and XGBoost also tried an absolute-error objective, which on log prices targets the median percentage error directly. Selection still uses the 2026-03 validation month only.
- **Fixed: every candidate is now scored on the same homes.** Before, each training window's own price cutoffs filtered the validation and test months, so a 3-month window was graded on 16 more extreme-price homes than a 24-month window, and models with different windows had slightly different test sets (`preprocessing.eval_set`).
- **Two new feature ideas tested and rejected on validation:** a market-trend feature (months since 2024-01) and the price level of the 10 nearest training sales. See section 2.

**Final model = the tuned model with the lowest validation MdAPE.** The test month (2026-04) plays no part in the choice.

**Run first:** `python src/models.py --tuned` (about 2 hours) and `python src/evaluate_final.py` (a few minutes). This notebook loads their outputs.

## 1. Setup

In [ ]:
import ast
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import evaluate_final as ef
import evaluation
import models
import preprocessing

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
LABELS, NAMES = ef.LABELS, ef.NAMES

clean = preprocessing.load_clean("../data/processed/sfr_clean.parquet")
metrics = ef.load_metrics()
final = ef.pick_final(metrics)
arts = ef.load_models()
test, preds = ef.test_predictions(clean, arts)
summary = pd.read_csv("../docs/metrics_summary.csv")
intervals = pd.read_csv("../docs/prediction_intervals.csv")
y, p = test["ClosePrice"], preds[final]
print(f"final model: {LABELS[final]}; {len(test):,} test homes (2026-04), scored by every model")

## 2. Choosing the final model on validation

Best validation MdAPE per model after the wider search, next to the Week 5/7 search (same `week6` features). The week 7 numbers are from the earlier scoring, which kept up to 16 more extreme-price homes in short windows; the difference is under 0.01 points for the chosen windows.

In [ ]:
val = metrics[metrics["stage"] == "validation (selection)"]
best = val.loc[val.groupby("model")["MdAPE"].idxmin(), ["model", "train_months", "params", "MdAPE", "MAPE", "R2"]].set_index("model")
before = pd.concat([pd.read_csv(f"../docs/metrics_{models.artifact_stem(n, 'week6')}.csv") for n in NAMES])
before = before[before["stage"] == "validation (selection)"].groupby("model")["MdAPE"].min()
best.insert(3, "MdAPE before (Weeks 5-7)", before)
best.loc[NAMES].rename(index=LABELS).sort_values("MdAPE")

In [ ]:
# Week 8 feature ideas, validation month only (docs/feature_set_comparison_week8.csv)
fs8 = pd.read_csv("../docs/feature_set_comparison_week8.csv")
fs8.pivot(index="feature_set", columns="model", values="MdAPE").rename(columns=LABELS).loc[
    ["week6", "week6+trend", "week6+neighbors", "week6+trend+neighbors"]]

## 3. Test results (2026-04), every model on the same homes

Linear Regression baseline next to the tree models (IDX Best Practices §10).

In [ ]:
cols = ["train_months", "R2", "MdAPE", "MAPE", "MAE", "RMSE", "within_10pct", "median_pct_error"]
overall = summary[(summary["stage"] == "test") & (summary["segment_type"] == "overall")].set_index("model")
table = overall.loc[NAMES, cols].assign(**{"model file (MB)": [arts[n]["file_mb"] for n in NAMES]})
table.rename(index=LABELS)

In [ ]:
# Paired bootstrap: is the final model's lead more than noise?
rows = [{"compared with": LABELS[n], **evaluation.paired_bootstrap_mdape(y, preds[n], p)} for n in NAMES if n != final]
pd.DataFrame(rows).rename(columns={"mdape_a": "other MdAPE", "mdape_b": f"{LABELS[final]} MdAPE"})

## 4. Stability over time: rolling backtest

In [ ]:
hist = metrics[metrics["stage"].isin(["test", "backtest"])]
hist.pivot(index="test_month", columns="model", values="MdAPE")[NAMES].rename(columns=LABELS)

In [ ]:
from IPython.display import Image
Image("../docs/figures/model_mdape_by_month.png")

## 5. Price bands

Quintiles of the actual close price in the test month. MdAPE = typical size of the error; median signed error = direction (> 0 means the model over-values the typical home in that band).

In [ ]:
bands = summary[(summary["stage"] == "test") & (summary["segment_type"] == "price_band")]
bands.pivot(index="segment", columns="model_label", values="MdAPE")[[LABELS[n] for n in NAMES]]

In [ ]:
fb = bands[bands["model"] == final].set_index("segment")[["n", "MdAPE", "MAPE", "MAE", "within_10pct", "median_pct_error"]]
display(fb)
Image("../docs/figures/final_model_by_price_band.png")

## 6. Geography: counties with at least 200 test homes

In [ ]:
county = summary[(summary["stage"] == "test") & (summary["segment_type"] == "county")]
county.pivot(index="segment", columns="model_label", values="MdAPE")[[LABELS[n] for n in NAMES]].sort_values(LABELS[final])

## 7. Other segments for the final model

In [ ]:
def by(groups, name, order=None):
    t = evaluation.metrics_by_group(y, p, groups).set_index("group")[["n", "MdAPE", "MAPE", "median_pct_error"]]
    return (t.loc[order] if order else t).rename_axis(name)

size_q = pd.qcut(test["LivingArea"], 5)
size_labels = [f"{b.left:,.0f}–{b.right:,.0f}" for b in size_q.cat.categories]
display(by(size_q.cat.rename_categories(size_labels).astype(str), "living area (sq ft)", size_labels))
age_labels = ["0–5", "6–20", "21–50", "51–80", "81+"]
age = pd.cut(test["property_age"], [-1, 5, 20, 50, 80, 500], labels=age_labels).astype(str)
display(by(age, "age at sale (years)", age_labels))
display(by(test["NewConstructionYN"].astype("object").fillna("missing").astype(str), "new construction"))
display(by(test["Latitude"].isna().map({True: "no coordinates", False: "has coordinates"}), "coordinates"))

## 8. Error distribution and the largest misses

In [ ]:
pct = 100 * (p - y) / y
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pct.clip(-60, 60), bins=120, color="#2a78d6")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Signed % error (clipped at ±60%)"); axes[0].set_xlabel("(predicted − actual) / actual, %")
axes[1].hexbin(y, p, gridsize=60, xscale="log", yscale="log", mincnt=1, bins="log", cmap="Blues")
lims = [y.min(), y.max()]
axes[1].plot(lims, lims, color="black", linewidth=0.8)
axes[1].set_title("Predicted vs actual close price (log scale)"); axes[1].set_xlabel("actual"); axes[1].set_ylabel("predicted")
plt.tight_layout(); plt.show()
print("share of homes within ±5% / ±10% / ±20%:",
      [f"{(pct.abs() <= t).mean():.1%}" for t in (5, 10, 20)])

In [ ]:
# What the worst 5% of predictions have in common (medians / shares), compared with the rest
worst = pct.abs() >= pct.abs().quantile(0.95)
def profile(d):
    return pd.Series({"homes": len(d), "median price": d["ClosePrice"].median(), "median sq ft": d["LivingArea"].median(),
                      "median age": d["property_age"].median(), "median lot sq ft": d["LotSizeSquareFeet"].median(),
                      "share missing lot size": d["LotSizeSquareFeet"].isna().mean(),
                      "share new construction": d["NewConstructionYN"].astype("float").mean(),
                      "share over-valued": (pct[d.index] > 0).mean()})
pd.DataFrame({"worst 5%": profile(test[worst]), "other 95%": profile(test[~worst])})

## 9. Prediction intervals

An AVM should say how sure it is. For each band of **predicted** price, the interval is the middle 80% (or 90%) of the final model's actual/predicted ratios on the validation month (model fit on the months before 2026-03). The coverage columns check those intervals on the test month: an honest 80% interval should contain about 80% of test homes.

In [ ]:
iv = intervals.copy()
for c in ["low_80", "high_80", "low_90", "high_90"]:
    iv[c] = (100 * iv[c]).round(1).astype(str) + "%"
for c in ["test_coverage_80", "test_coverage_90"]:
    iv[c] = (100 * iv[c]).round(1)
iv["predicted price"] = [f"${lo / 1e3:,.0f}k – " + ("" if np.isinf(hi) else f"${hi / 1e3:,.0f}k")
                         for lo, hi in zip(intervals["predicted_low"], intervals["predicted_high"])]
iv[["band", "predicted price", "val_n", "test_n", "low_80", "high_80", "test_coverage_80", "low_90", "high_90", "test_coverage_90"]]

## 10. Findings

**Final model: LightGBM** (24-month window, 511 leaves, minimum 10 homes per leaf, absolute-error objective). It had the lowest validation MdAPE (6.78%), essentially tied with XGBoost (6.79%).

| Test 2026-04, same 12,328 homes | R² | MdAPE | MAPE | MAE | Within 10% | Model file |
|---|---|---|---|---|---|---|
| Linear Regression (baseline) | 0.827 | 10.99% | 15.1% | $205k | 46% | 0.1 MB |
| Decision Tree | 0.845 | 9.32% | 13.8% | $189k | 53% | 0.7 MB |
| Random Forest | 0.897 | 7.52% | 11.1% | $153k | 61% | 1,106 MB |
| **LightGBM (final)** | **0.912** | **6.96%** | **10.5%** | **$142k** | **64%** | 39 MB |
| XGBoost | 0.905 | 7.15% | 10.8% | $147k | 63% | 17 MB |

- **The wider tuning helped the boosted models only.** LightGBM's test MdAPE dropped from 7.15% (Week 7) to 6.96%; the absolute-error objective, which targets the median error directly, was the main gain. The Decision Tree and Random Forest picked the same settings as before.
- **More than noise:** LightGBM beats the baseline by 4.0 MdAPE points (95% paired-bootstrap interval 3.8–4.3) and the Random Forest by 0.56 (0.42–0.69).
- **Stable over time:** LightGBM's MdAPE is 6.8–7.0% in all four months, the lowest of every model in each.
- **Typical accuracy:** 38% of test homes within ±5% of the sale price, 64% within ±10%, 87% within ±20%.
- **Price bands:** most accurate for $575k–$1.1M homes (MdAPE 5.3–6.0%). Entry-level homes are over-valued by 2.4% and luxury homes (above $1.67M) under-valued by 5.0% (MdAPE 10.2%), still the weakest segment but far better than the baseline (−9.6%, 15.7%).
- **Geography:** LightGBM or XGBoost is best in every large county (LightGBM in 6 of 10; XGBoost is slightly lower in San Bernardino, Orange, San Luis Obispo and Alameda). Inland Empire is easiest (Riverside 5.6%); Alameda is hardest (9.8%).
- **Other segments:** errors grow with age (4.5% for homes 0–5 years old, 10.5% for homes over 80) and size (8.3% for the largest fifth). The 81 homes without coordinates have MdAPE 13.2%.
- **The worst 5% of predictions** are mostly over-valuations (75%) of older homes (median age 61 vs 49). The likely cause is condition (fixer-uppers, distressed sales), which the MLS fields used here do not describe.
- **Prediction intervals are honest:** the 80% ranges built on the validation month contain 78–81% of test homes in every price band, and the 90% ranges 88–90%. They are wider upward for expensive homes (up to +26% at 80%), so the app can show e.g. "$1.2M (likely $1.0M–$1.5M)".

**Week 8 feature ideas (validation only):** the market-trend feature made every model worse. The price level of the 10 nearest sales cut the Linear Regression's MdAPE from 10.8% to 9.3% but did not help the boosted models, which already learn location from ZIP code, city and coordinates; it is not in the final model.

### Mistakes found and fixed this week
- **Candidates were scored on slightly different homes.** Each training window's own price cutoffs filtered the validation and test months, so short windows were graded on up to 16 more extreme-price homes, and models with different windows had different test sets. Now every candidate is scored on the same homes (`preprocessing.eval_set`). Week 5–7 numbers changed by less than 0.01 points for the chosen windows.
- **Grid edges:** the Week 5 and 7 choices sat on the edge of their search grids; the wider search confirmed the tree models and improved the boosted ones.

### Limitations
- **The 2026-03 backtest month is also the validation month**, so it is not an independent check; 2026-01 and 2026-02 are.
- **Cleaning rules that use the sale price** (close/list ratio, price per sq ft, minimum price) also apply to the test month. They remove 359 sales (0.12%), mostly typos, so real-world error on unfiltered data would be slightly higher.
- **One market, one period:** CRMLS sales from January 2024 to April 2026 (Southern and Central California, parts of the Bay Area). Accuracy elsewhere is unknown.
- **No condition or renovation data**, the most likely source of the largest errors.
- **School-district boundaries are the 2024-25 snapshot**, applied to 2024–2026 sales.

**Next (Week 9, optional):** a Streamlit app that loads the saved LightGBM model, looks up the school district from coordinates and shows the price with the 80% range.